# 2Day Start 役職推定パイプライン実行ノートブック
**2日目開始時点でのデータを使用して役職を推定するパイプライン**

このノートブックは`run_train_pipeline.py`の構造を参考にしながら、2日目開始シナリオに対応させたものです。

このノートブックで行うこと:
- 実行環境とパスの設定
- 既存モジュールの import と自動リロード
- 2日目開始時のデータ準備（day==1フィルタ、脱落者データ除外）
- 役職割り当てロジックを用いた訓練・評価
- 特徴量重要度と SHAP による解釈可能性分析
- CSV/画像の保存と結果確認

注意:
- このパイプラインは **day==1** のデータのみを使用します
- 2日目の脱落者情報（exec_id, attack_id）を基にした制約を考慮します

## 1. Notebookで使う実行環境・パス設定

Python バージョン確認、ライブラリ import、プロジェクトルートと `sys.path` を設定します。

In [1]:
import os
import sys
import json
import time
import traceback
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version}")

# notebooks/ から見たプロジェクトルート
PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"
DATA_DIR = PROJECT_ROOT / "data"

# パスを sys.path に追加
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SRC_PATH: {SRC_PATH}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"CONFIG_PATH exists: {CONFIG_PATH.exists()}")
print(f"DATA_DIR: {DATA_DIR}")

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
PROJECT_ROOT: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub
SRC_PATH: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\src
MODELS_DIR: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\models
CONFIG_PATH exists: True
DATA_DIR: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\data


## 2. 既存 `.py` のインポートと自動リロード設定

モジュール更新を即時反映するため、autoreload を有効化します。

In [2]:
%load_ext autoreload
%autoreload 2

# プロジェクトルートとsrcディレクトリの絶対パスをsys.pathに追加
PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"
if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))
from src.pipelines.training_pipeline import load_training_config, get_data_paths, get_models_dir

print("Module import complete.")

Module import complete.


C:\Users\takic\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 設定読み込みと2Day Start用パラメータ管理

設定ファイルを読み込み、day_filter=1（1日目データ）に固定して2日目開始時のシナリオに対応させます。

In [3]:
RUN_OPTIONS = {
    "run_training": True,
    "top_k_features": 10,
    "show_only_required_artifacts": True,
    "day_filter": 1,  # 2Day Start用: 1日目データを使用
    "day2_flag": True,  # 2日目脱落者制約を有効
}

config = load_training_config(CONFIG_PATH)

# 2Day Start用に設定をオーバーライド
override = {
    "n_trials": 30,  # 探索回数を10回に固定（短時間実行用）
    "day_filter": 1,  # 2Day Startなので1固定
}

# 見た目と実際の挙動を一致させるため、config側にも反映
config["n_trials"] = int(override["n_trials"])

print("="*70)
print("2DAY START パイプライン設定")
print("="*70)
print("\nRun options:")
print(json.dumps(RUN_OPTIONS, ensure_ascii=False, indent=2))
print("\nTraining config (effective):")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("\n2Day Start override:")
print(json.dumps(override, ensure_ascii=False, indent=2))
print(f"\nEffective n_trials: {config['n_trials']}")
print("\nFeature policy:")
if bool(config.get("lang_feature", False)):
    print("- lang_feature=True ですが、話者推定モデル/話者推定確率特徴は無効です。")
else:
    print("- 話者推定モデル/話者推定確率特徴は使用しません。")

2DAY START パイプライン設定

Run options:
{
  "run_training": true,
  "top_k_features": 10,
  "show_only_required_artifacts": true,
  "day_filter": 1,
  "day2_flag": true
}

Training config (effective):
{
  "data_paths": [
    "data/2025spring/all_feature_table_2025sp17_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_with_talks2.csv"
  ],
  "data_paths_day2": [
    "data/2025spring/all_feature_table_2025sp17_day2_with_talks.csv",
    "data/2025summer/all_feature_table_2025summer_day2_with_talks.csv"
  ],
  "day_filter": 1,
  "lang_feature": false,
  "group_column": "source_file",
  "cv_folds": 5,
  "test_size": 0.2,
  "n_trials": 30,
  "leakage_drop_columns": [
    "True_Div_recepient_id_1",
    "True_Div_result_1",
    "True_Div_recepient_id_2",
    "True_Div_result_2",
    "target_total_votes",
    "alive"
  ]
}

2Day Start override:
{
  "n_trials": 30,
  "day_filter": 1
}

Effective n_trials: 30

Feature policy:
- 話者推定モデル/話者推定確率特徴は使用しません。


## 4. 2Day Start用データ準備

day==1 のデータのみを読み込み、脱落者情報を基にした制約を準備します。

In [4]:
# データのロードと前処理
def create_perspective_dataset(df_raw, meta_info):
    """
    盤面情報（誰が死んだか、占われたか）をNLP特徴量に付加する
    """
    df = df_raw.copy()

    # 1. 試合ごとの「死者の情報」をマッピングするための辞書を作成
    game_map = df.groupby('source_file').first()[['attack_id', 'exec_id']].to_dict('index')
    
    # 占い結果のログ（誰が誰を占ったか）を抽出
    div_map = {}
    for i in range(len(meta_info['source_file'])):
        g = meta_info['source_file'][i]
        owner_id = df.iloc[i]['id'] # 占い師本人
        target_id = meta_info['div_id1'][i] # 占われた人
        res = meta_info['div_result1'][i] # 結果
        if g not in div_map: div_map[g] = {}
        div_map[g][owner_id] = (target_id, res)

    # 誰が誰を疑っていたか・誰に投票したかの情報を集約
    game_victim_actions = {}

    for game_id in df['source_file'].unique():
        g_data = df[df['source_file'] == game_id]

        atk_id = g_data['attack_id'].iloc[0]
        exe_id = g_data['exec_id'].iloc[0]
        actions = {
            'atk_voted': g_data.loc[g_data['id'] == atk_id, 'day1_vote_id'].values[0] if atk_id in g_data['id'].values else None,
            'atk_suspected': g_data.loc[g_data['id'] == atk_id, 'Sus_recepient_id'].values[0] if atk_id in g_data['id'].values else None,
            'exe_voted': g_data.loc[g_data['id'] == exe_id, 'day1_vote_id'].values[0] if exe_id in g_data['id'].values else None,
            'exe_suspected': g_data.loc[g_data['id'] == exe_id, 'Sus_recepient_id'].values[0] if exe_id in g_data['id'].values else None,
        }
        game_victim_actions[game_id] = actions
    new_features = []
    for i, row in df.iterrows():
        game_id = row['source_file']
        agent_id = row['id']
        summary = game_map[game_id]
        # --- 共通盤面特徴量 (どの視点からも見える事実) ---
        is_attacked = 1 if agent_id == summary['attack_id'] else 0
        is_executed = 1 if agent_id == summary['exec_id'] else 0
        divined_by_victim_black = 0
        divined_by_victim_white = 0
        victim_id = summary['attack_id']
        if game_id in div_map and victim_id in div_map[game_id]:
            target, res = div_map[game_id][victim_id]
            if target == agent_id:
                if str(res) in ["WEREWOLF", "人狼", "黒"]: divined_by_victim_black = 1
                if str(res) in ["HUMAN", "白", "not(人狼)"]: divined_by_victim_white = 1

        divined_by_exe_black = 0
        divined_by_exe_white = 0
        exe_id = summary['exec_id']
        if game_id in div_map and exe_id in div_map[game_id]:
            target, res = div_map[game_id][exe_id]
            if target == agent_id:
                if str(res) in ["WEREWOLF", "人狼", "黒"]: divined_by_exe_black = 1
                if str(res) in ["HUMAN", "白", "not(人狼)"]: divined_by_exe_white = 1
        is_target_of_atk = 0
        if agent_id == actions.get('atk_voted') or agent_id == actions.get('atk_suspected'):
            is_target_of_atk = 1
        is_vote_of_exe,is_suspected_of_exe = 0,0
        if agent_id == actions.get('exe_voted'):
            is_vote_of_exe = 1
        if agent_id == actions.get('exe_suspected'):
            is_suspected_of_exe = 1
        is_vote_of_atk,is_suspected_of_atk = 0,0
        if agent_id == actions.get('atk_voted'):
            is_vote_of_atk = 1
        if agent_id == actions.get('atk_suspected'):
            is_suspected_of_atk = 1
        new_features.append({
            'feat_is_attacked': is_attacked,
            'feat_is_executed': is_executed,
            'feat_divined_by_victim_black': divined_by_victim_black,
            'feat_divined_by_victim_white': divined_by_victim_white,
            'feat_divined_by_exe_black': divined_by_exe_black,
            'feat_divined_by_exe_white': divined_by_exe_white,
            'feat_is_vote_of_exe': is_vote_of_exe,
            'feat_is_suspected_of_exe': is_suspected_of_exe,
            'feat_is_vote_of_atk': is_vote_of_atk,
            'feat_is_suspected_of_atk': is_suspected_of_atk
            })
    feat_df = pd.DataFrame(new_features)
    return pd.concat([df.reset_index(drop=True), feat_df], axis=1)

In [5]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import joblib

print("="*70)
print("2DAY START データ準備")
print("="*70)

# データパスの取得
data_paths = get_data_paths(config, data_type="day1")

# CSVの読み込みと day==1 フィルタ
dfs = []
for p in data_paths:
    p_path = Path(p)  # 文字列をPathオブジェクトに変換
    df_raw = pd.read_csv(p)
    df_day1 = df_raw[df_raw["day"] == RUN_OPTIONS["day_filter"]].copy()
    df_day1['dataset_tag'] =  p_path.stem
    dfs.append(df_day1)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"\n✓ Combined shape: {df.shape}")

# attack_id が NaN のサンプルを除外
df_filtered = df.dropna(subset=["attack_id"]).copy()
print(f"✓ After dropping NaN attack_id: {df_filtered.shape}")

# SEER で True_Div_result_1 が NaN のファイルを除外
if "True_Div_result_1" in df_filtered.columns:
    del_paths = df_filtered[(df_filtered["role"]=="SEER") & (df_filtered["True_Div_result_1"].isna())]["source_file"].unique()
    if len(del_paths) > 0:
        print(f"  Excluding {len(del_paths)} files with missing True_Div_result_1 for SEER")
        df_filtered = df_filtered[~df_filtered["source_file"].isin(del_paths)]

# 目的変数のエンコード
label_encoder = LabelEncoder()
df_filtered['role_encoded'] = label_encoder.fit_transform(df_filtered['role'].astype(str))
y = df_filtered['role_encoded'].values

# メタ情報の抽出
meta = {
    'source_file': df_filtered['source_file'].values,
    'dataset_tag': df_filtered['dataset_tag'].values if 'dataset_tag' in df_filtered.columns else np.array(["unknown"] * len(df_filtered)),
    'div_result1': df_filtered['True_Div_result_1'].values if 'True_Div_result_1' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_id1': df_filtered['True_Div_recepient_id_1'].values if 'True_Div_recepient_id_1' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_result2': df_filtered['True_Div_result_2'].values if 'True_Div_result_2' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'div_id2': df_filtered['True_Div_recepient_id_2'].values if 'True_Div_recepient_id_2' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'exec_id': df_filtered['exec_id'].values if 'exec_id' in df_filtered.columns else np.full(len(df_filtered), np.nan),
    'attack_id': df_filtered['attack_id'].values if 'attack_id' in df_filtered.columns else np.full(len(df_filtered), np.nan),
}

# 特徴量行列の作成
# = df_filtered['combined_text'].astype(str).values
agent_names = df_filtered['agent_name'].astype(str).values

base_drop_cols = [
    'id', 'role', 'source_file', 'dataset_tag', 'day', 'role_encoded', 'role_encoded',
    'Est_id_Fact_role', 'Est_id_Est_roles', 'character_name','agent_name', 'combined_text', 'seer_co_order'
]
meta_raw_cols = [
    'True_Div_result_1', 'True_Div_recepient_id_1',
    'True_Div_result_2', 'True_Div_recepient_id_2',
    'exec_id', 'attack_id'
]
day2_col = [c for c in df_filtered.columns if c.startswith('day2_')or c.startswith('day1_')]
id_like_cols = [c for c in df_filtered.columns if c.endswith('_id')]
flag_like_cols = [c for c in df_filtered.columns if c.endswith('_flag')]

all_drop_cols = list(set(base_drop_cols + meta_raw_cols + id_like_cols + flag_like_cols + day2_col))
# --- 2. 特徴量エンジニアリング（dropする前のきれいなデータで実行） ---
# X_df を作るのではなく、元の df_filtered を渡します
X_df_new_dataset = create_perspective_dataset(df_filtered, meta)

# --- 3. 最後に不要なカラムを一括削除 ---
# 特徴量生成が終わった後に、学習に使わない文字列やメタ情報を消します
X_df_final = X_df_new_dataset.drop(columns=all_drop_cols, errors='ignore')

# 特徴量リストの保存
final_features = X_df_final.columns.tolist()

# 数値化
X = X_df_final.apply(pd.to_numeric, errors='coerce').fillna(0).values

print(f"\n✓ Data shapes:")
print(f"  X: {X.shape}")
display(pd.Series(meta['dataset_tag']).value_counts().rename_axis('dataset').reset_index(name='samples'))

2DAY START データ準備
✓ Data files selected:
  - all_feature_table_2025sp17_with_talks.csv
  - all_feature_table_2025summer_with_talks2.csv

✓ Combined shape: (1125, 232)
✓ After dropping NaN attack_id: (745, 232)
  Excluding 7 files with missing True_Div_result_1 for SEER

✓ Data shapes:
  X: (710, 178)


,dataset,samples
0,all_feature_table_2025summer_with_talks2,375
1,all_feature_table_2025sp17_with_talks,335


## 5. 役職割り当て関数（制約付き予測）

SEER視点、WEREWOLF視点、VILLAGER視点などの異なる視点から役職を割り当てます。
脱落者（exec_id, attack_id）を基にした制約を組み込みます。

In [6]:
import itertools
import torch

def assign_roles_for_non_seer(
    logits, y_batch, role_counts, role_names, label_encoder,
    fixed_self_role_name, exec_id_batch, attack_id_batch, day2_flag=True, debug=False):
    """
    非SEER視点（VILLAGER/POSSESSED/WEREWOLF）での役職割り当て
    脱落者を人狼ではなく設定
    """
    role_names = list(role_names)
    if isinstance(logits, torch.Tensor): 
        logits = logits.detach().cpu().numpy()
    if isinstance(y_batch, torch.Tensor): 
        y_batch = y_batch.detach().cpu().numpy()
    
    num_players = logits.shape[0]
    if num_players != 5:
        return np.array([]), np.array([])
    
    pred_list, y_list = [], []

    for self_index in range(num_players):
        self_role = y_batch[self_index]
        self_role_name = label_encoder.inverse_transform([self_role])[0]
        
        if self_role_name != fixed_self_role_name:
            continue

        if day2_flag and (self_index + 1 in exec_id_batch or self_index + 1 in attack_id_batch):
            if debug:
                print(f"skip id:{self_index + 1} exec or attack")
            continue

        other_indices = [i for i in range(num_players) if i != self_index]
        logits_others = logits[other_indices]
        y_others = y_batch[other_indices]
        
        non_werewolf_indices_in_others = []
        if day2_flag:
            exec_id = exec_id_batch[self_index]
            attack_id = attack_id_batch[self_index]
            if np.isnan(attack_id) or np.isnan(exec_id): 
                continue
            attack_id_idx = int(attack_id) - 1
            exec_id_idx = int(exec_id) - 1
            if self_index == exec_id_idx or self_index == attack_id_idx: 
                continue
            if exec_id_idx in other_indices:
                non_werewolf_indices_in_others.append(other_indices.index(exec_id_idx))
            if attack_id_idx in other_indices:
                non_werewolf_indices_in_others.append(other_indices.index(attack_id_idx))
            non_werewolf_indices_in_others = list(set(non_werewolf_indices_in_others))
        
        reduced_counts = role_counts.copy()
        if reduced_counts.get(fixed_self_role_name, 0) == 0:
            continue
        reduced_counts[fixed_self_role_name] -= 1
        roles_pool = [role for role, count in reduced_counts.items() for _ in range(count)]
        if len(roles_pool) != 4: 
            continue
        
        best_perm = None
        best_score = -np.inf
        for perm in set(itertools.permutations(roles_pool)):
            if any(perm[idx] == "WEREWOLF" for idx in non_werewolf_indices_in_others):
                continue
            score = sum(np.log(logits_others[i][role_names.index(role)] + 1e-9) for i, role in enumerate(perm))
            if score > best_score:
                best_score, best_perm = score, perm
        
        if best_perm:
            pred_encoded = label_encoder.transform(list(best_perm))
            pred_list.append(pred_encoded)
            y_list.append(y_others)

    return (np.concatenate(pred_list) if pred_list else np.array([]),
            np.concatenate(y_list) if y_list else np.array([]))


def assign_roles_for_seer_by_divination(
    logits, y_batch, role_counts, role_names, label_encoder,
    div_result_array1, div_id_array1, div_result_array2=None, div_id_array2=None,
    exec_id_batch=None, attack_id_batch=None, day2_flag=True, debug=False):
    """
    SEER視点での役職割り当て
    占い結果を基に黒（人狼）/白（人狼以外）を分類
    """
    role_names = list(role_names)
    if isinstance(logits, torch.Tensor): 
        logits = logits.detach().cpu().numpy()
    if isinstance(y_batch, torch.Tensor): 
        y_batch = y_batch.detach().cpu().numpy()
    
    num_players = logits.shape[0]
    if num_players != 5:
        empty_result = (np.array([]), np.array([]))
        return {"black": empty_result, "white": empty_result}

    pred_list_black, y_list_black = [], []
    pred_list_white, y_list_white = [], []

    for self_index in range(num_players):
        self_role_name = label_encoder.inverse_transform([y_batch[self_index]])[0]

        if self_role_name != "SEER":
            continue

        if day2_flag and (self_index+1 in exec_id_batch or self_index+1 in attack_id_batch):
            continue

        other_indices = [i for i in range(num_players) if i != self_index]
        logits_others = logits[other_indices]
        y_others = y_batch[other_indices]

        non_werewolf_indices_in_others = []
        if day2_flag and exec_id_batch is not None and attack_id_batch is not None:
            exec_id = exec_id_batch[self_index]
            attack_id = attack_id_batch[self_index]
            if not (np.isnan(exec_id) or np.isnan(attack_id)):
                exec_id_idx = int(exec_id) - 1
                attack_id_idx = int(attack_id) - 1
                if exec_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(exec_id_idx))
                if attack_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(attack_id_idx))
        
        div_constraints = {}
        all_div_results = [(div_result_array1, div_id_array1)]
        if day2_flag and div_result_array2 is not None and div_id_array2 is not None:
            all_div_results.append((div_result_array2, div_id_array2))

        first_div_result_for_classification = None

        for i, (div_result_array, div_id_array) in enumerate(all_div_results):
            if np.isnan(div_id_array[self_index]): 
                continue
            
            div_id = int(div_id_array[self_index]) - 1
            if div_id not in other_indices: 
                continue
            
            div_target_index_in_others = other_indices.index(div_id)
            div_result_str = str(div_result_array[self_index]).strip()

            if div_result_str in ["WEREWOLF", "人狼", "黒"]:
                div_constraints[div_target_index_in_others] = "WEREWOLF"
                if i == 0: 
                    first_div_result_for_classification = "WEREWOLF"
            elif div_result_str in ["HUMAN", "白", "not(人狼)"]:
                div_constraints[div_target_index_in_others] = "HUMAN"
                if i == 0: 
                    first_div_result_for_classification = "HUMAN"
        
        if first_div_result_for_classification is None:
            continue

        reduced_counts = role_counts.copy()
        if reduced_counts.get("SEER", 0) == 0: 
            continue
        reduced_counts["SEER"] -= 1
        roles_pool = [role for role, count in reduced_counts.items() for _ in range(count)]
        if len(roles_pool) != 4: 
            continue

        best_score, best_perm = -np.inf, None

        for perm in set(itertools.permutations(roles_pool)):
            is_valid = True
            
            if any(perm[idx] == "WEREWOLF" for idx in non_werewolf_indices_in_others):
                continue
            
            for target_idx, result in div_constraints.items():
                if result == "WEREWOLF" and perm[target_idx] != "WEREWOLF":
                    is_valid = False; break
                if result == "HUMAN" and perm[target_idx] == "WEREWOLF":
                    is_valid = False; break
            if not is_valid: 
                continue

            score = sum(np.log(logits_others[i][role_names.index(role)] + 1e-9) for i, role in enumerate(perm))
            if score > best_score:
                best_score, best_perm = score, perm

        if best_perm:
            pred_encoded = label_encoder.transform(list(best_perm))
            if first_div_result_for_classification == "WEREWOLF":
                pred_list_black.append(pred_encoded)
                y_list_black.append(y_others)
            elif first_div_result_for_classification == "HUMAN":
                pred_list_white.append(pred_encoded)
                y_list_white.append(y_others)

    result = {
        "black": (np.concatenate(pred_list_black) if pred_list_black else np.array([]),
                  np.concatenate(y_list_black) if y_list_black else np.array([])),
        "white": (np.concatenate(pred_list_white) if pred_list_white else np.array([]),
                  np.concatenate(y_list_white) if y_list_white else np.array([])),
    }
    return result

print("✓ 役職割り当て関数が定義されました")

✓ 役職割り当て関数が定義されました


In [7]:
# --- 修正版: SEER割り当てロジックを緩和したバージョン ---
def assign_roles_for_seer_by_divination_relaxed(
    logits, y_batch, role_counts, role_names, label_encoder,
    div_result_array1, div_id_array1, div_result_array2=None, div_id_array2=None,
    exec_id_batch=None, attack_id_batch=None, day2_flag=True, debug=False):
    """
    SEER視点での役職割り当て（占い結果が1つでもあれば割り当てる）
    """
    role_names = list(role_names)
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu().numpy()
    if isinstance(y_batch, torch.Tensor):
        y_batch = y_batch.detach().cpu().numpy()
    num_players = logits.shape[0]
    if num_players != 5:
        empty_result = (np.array([]), np.array([]))
        return {"black": empty_result, "white": empty_result}
    pred_list_black, y_list_black = [], []
    pred_list_white, y_list_white = [], []
    for self_index in range(num_players):
        self_role_name = label_encoder.inverse_transform([y_batch[self_index]])[0]
        if self_role_name != "SEER":
            continue
        if day2_flag and (self_index+1 in exec_id_batch or self_index+1 in attack_id_batch):
            continue
        other_indices = [i for i in range(num_players) if i != self_index]
        logits_others = logits[other_indices]
        y_others = y_batch[other_indices]
        non_werewolf_indices_in_others = []
        if day2_flag and exec_id_batch is not None and attack_id_batch is not None:
            exec_id = exec_id_batch[self_index]
            attack_id = attack_id_batch[self_index]
            if not (np.isnan(exec_id) or np.isnan(attack_id)):
                exec_id_idx = int(exec_id) - 1
                attack_id_idx = int(attack_id) - 1
                if exec_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(exec_id_idx))
                if attack_id_idx in other_indices:
                    non_werewolf_indices_in_others.append(other_indices.index(attack_id_idx))
        div_constraints = {}
        all_div_results = [(div_result_array1, div_id_array1)]
        if day2_flag and div_result_array2 is not None and div_id_array2 is not None:
            all_div_results.append((div_result_array2, div_id_array2))
        first_div_result_for_classification = None
        found_div_result = False
        for i, (div_result_array, div_id_array) in enumerate(all_div_results):
            if np.isnan(div_id_array[self_index]):
                continue
            div_id = int(div_id_array[self_index]) - 1
            if div_id not in other_indices:
                continue
            div_target_index_in_others = other_indices.index(div_id)
            div_result_str = str(div_result_array[self_index]).strip()
            if div_result_str in ["WEREWOLF", "人狼", "黒"]:
                div_constraints[div_target_index_in_others] = "WEREWOLF"
                if first_div_result_for_classification is None:
                    first_div_result_for_classification = "WEREWOLF"
                found_div_result = True
            elif div_result_str in ["HUMAN", "白", "not(人狼)"]:
                div_constraints[div_target_index_in_others] = "HUMAN"
                if first_div_result_for_classification is None:
                    first_div_result_for_classification = "HUMAN"
                found_div_result = True
        # 1つでも占い結果があれば割り当てる（なければスキップ）
        if not found_div_result:
            continue
        reduced_counts = role_counts.copy()
        if reduced_counts.get("SEER", 0) == 0:
            continue
        reduced_counts["SEER"] -= 1
        roles_pool = [role for role, count in reduced_counts.items() for _ in range(count)]
        if len(roles_pool) != 4:
            continue
        best_score, best_perm = -np.inf, None
        for perm in set(itertools.permutations(roles_pool)):
            is_valid = True
            if any(perm[idx] == "WEREWOLF" for idx in non_werewolf_indices_in_others):
                continue
            for target_idx, result in div_constraints.items():
                if result == "WEREWOLF" and perm[target_idx] != "WEREWOLF":
                    is_valid = False; break
                if result == "HUMAN" and perm[target_idx] == "WEREWOLF":
                    is_valid = False; break
            if not is_valid:
                continue
            score = sum(np.log(logits_others[i][role_names.index(role)] + 1e-9) for i, role in enumerate(perm))
            if score > best_score:
                best_score, best_perm = score, perm
        if best_perm:
            pred_encoded = label_encoder.transform(list(best_perm))
            if first_div_result_for_classification == "WEREWOLF":
                pred_list_black.append(pred_encoded)
                y_list_black.append(y_others)
            elif first_div_result_for_classification == "HUMAN":
                pred_list_white.append(pred_encoded)
                y_list_white.append(y_others)
    result = {
        "black": (np.concatenate(pred_list_black) if pred_list_black else np.array([]),
                  np.concatenate(y_list_black) if y_list_black else np.array([])),
        "white": (np.concatenate(pred_list_white) if pred_list_white else np.array([]),
                  np.concatenate(y_list_white) if y_list_white else np.array([])),
    }
    return result

print("✓ 修正版 assign_roles_for_seer_by_divination_relaxed を定義しました")

✓ 修正版 assign_roles_for_seer_by_divination_relaxed を定義しました


## 6. モデルの訓練・最適化（StratifiedGroupKFold with CV）

4つの視点モデル（SEER, WEREWOLF, VILLAGER, POSSESSED）をOptuna で最適化し、
StratifiedGroupKFold によるクロスバリデーションで評価します。

In [8]:
def evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val):
    all_p, all_t = [], []
    num_games = len(y_val) // 5

    for i in range(num_games):
        s, e = i * 5, (i + 1) * 5
        if view_name == "SEER":
            # 修正版ロジックを利用
            res = assign_roles_for_seer_by_divination(
                    preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                    meta_val['div_result1'][s:e], meta_val['div_id1'][s:e],
                    meta_val['div_result2'][s:e], meta_val['div_id2'][s:e],
                    meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                    RUN_OPTIONS["day2_flag"]
            )
            for k in ["black", "white"]:
                if res[k][0].size > 0:
                    all_p.extend(res[k][0])
                    all_t.extend(res[k][1])
        else:
            p, t = assign_roles_for_non_seer(
                    preds_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                    view_name, meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                    RUN_OPTIONS["day2_flag"]
            )
            if p.size > 0:
                all_p.extend(p)
                all_t.extend(t)

    if len(all_t) == 0:
            return 0.0, 0

    target_label = 'POSSESSED' if view_name == 'WEREWOLF' else 'WEREWOLF'
    target_id = label_encoder.transform([target_label])[0]
    score = f1_score(all_t, all_p, labels=[target_id], average='macro', zero_division=0)
    return float(score), len(all_t)


In [9]:
# ==========================
# モデル訓練・交差検証（2day_voteと同じfold分割ロジックに修正）
# ==========================
import time
import optuna
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import json
import traceback
from collections import Counter
t0 = time.time()
models = {}
cv_results = []
run_status = {"ok": False, "error": None, "elapsed_sec": None}
try:
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    weight_dict = dict(zip(classes, weights))
    n_trials = 30
    # --- 2day_voteと同じfold分割ロジック ---
    groups = np.array([f"{d}::{s}" for d, s in zip(meta['dataset_tag'], meta['source_file'])])
    role_counts = {'POSSESSED': 1, 'SEER': 1, 'VILLAGER': 2, 'WEREWOLF': 1}
    class_names = list(label_encoder.classes_)
    n_splits = 5
    fold_seed = 42
    fold_search_iter = 50
    group_keys = np.array(groups)
    unique_games = np.unique(group_keys)
    dataset_tags = np.array(meta['dataset_tag'])
    agent_names_arr = agent_names if isinstance(agent_names, np.ndarray) else np.array(agent_names)
    # 1ゲームごとのインデックスと集計情報
    game_indices = {}
    game_agent_counter = {}
    game_dataset_counter = {}
    game_pair_counter = {}
    all_agents = set()
    all_datasets = set()
    all_pairs = set()
    for g in unique_games:
        idx = np.where(group_keys == g)[0]
        game_indices[g] = idx
        agent_cnt = Counter(agent_names_arr[i] for i in idx)
        ds_cnt = Counter(dataset_tags[i] for i in idx)
        pair_cnt = Counter((agent_names_arr[i], int(y[i])) for i in idx)
        game_agent_counter[g] = agent_cnt
        game_dataset_counter[g] = ds_cnt
        game_pair_counter[g] = pair_cnt
        all_agents.update(agent_cnt.keys())
        all_datasets.update(ds_cnt.keys())
        all_pairs.update(pair_cnt.keys())
    all_agents = sorted(all_agents)
    all_datasets = sorted(all_datasets)
    all_pairs = sorted(all_pairs)
    def ratio_mse_after_add(fold_counters, add_counter, keys, target_fold):
        total_mse = 0.0
        eps = 1e-12
        ideal = 1.0 / n_splits
        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counters[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            vals = np.array(vals, dtype=float)
            s = vals.sum()
            if s <= eps:
                continue
            ratios = vals / s
            total_mse += float(np.mean((ratios - ideal) ** 2))
        return total_mse / max(len(keys), 1)
    def variance_after_add(fold_counts, add_counter, keys, target_fold):
        total_var = 0.0
        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counts[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            total_var += float(np.var(vals))
        return total_var / max(len(keys), 1)
    def greedy_assign_once(seed: int):
        rng = np.random.default_rng(seed)
        game_order = list(unique_games)
        rng.shuffle(game_order)
        fold_agent_counts = [Counter() for _ in range(n_splits)]
        fold_dataset_counts = [Counter() for _ in range(n_splits)]
        fold_pair_counts = [Counter() for _ in range(n_splits)]
        fold_game_counts = [0 for _ in range(n_splits)]
        game_to_fold = {}
        w_dataset = 1.4
        w_agent = 1.2
        w_pair = 0.35
        w_size = 0.2
        def score_fold_with_candidate(fold_idx, game_id):
            ds_score = ratio_mse_after_add(fold_dataset_counts, game_dataset_counter[game_id], all_datasets, fold_idx)
            agent_score = ratio_mse_after_add(fold_agent_counts, game_agent_counter[game_id], all_agents, fold_idx)
            pair_score = variance_after_add(fold_pair_counts, game_pair_counter[game_id], all_pairs, fold_idx)
            size_vec = np.array(fold_game_counts, dtype=float)
            size_vec[fold_idx] += 1.0
            size_imbalance = float(np.var(size_vec))
            return (w_dataset * ds_score + w_agent * agent_score + w_pair * pair_score + w_size * size_imbalance)
        for g in game_order:
            best_fold = 0
            best_score = np.inf
            for f in range(n_splits):
                s = score_fold_with_candidate(f, g)
                if s < best_score:
                    best_score = s
                    best_fold = f
            game_to_fold[g] = best_fold
            fold_game_counts[best_fold] += 1
            fold_agent_counts[best_fold].update(game_agent_counter[g])
            fold_dataset_counts[best_fold].update(game_dataset_counter[g])
            fold_pair_counts[best_fold].update(game_pair_counter[g])
        final_score = 0.0
        final_score += w_dataset * ratio_mse_after_add(fold_dataset_counts, Counter(), all_datasets, 0)
        final_score += w_agent * ratio_mse_after_add(fold_agent_counts, Counter(), all_agents, 0)
        final_score += w_pair * variance_after_add(fold_pair_counts, Counter(), all_pairs, 0)
        final_score += w_size * float(np.var(np.array(fold_game_counts, dtype=float)))
        return game_to_fold, fold_pair_counts, fold_game_counts, final_score
    best = None
    best_score = np.inf
    for k in range(fold_search_iter):
        out = greedy_assign_once(seed=fold_seed + k)
        if out[3] < best_score:
            best = out
            best_score = out[3]
    game_to_fold, fold_pair_counts, fold_game_counts, _ = best
    fold_id_per_row = np.array([game_to_fold[g] for g in group_keys], dtype=int)
    split_indices = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        train_idx = np.where(fold_id_per_row != fold)[0]
        split_indices.append((train_idx, val_idx))

        fold_summary_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "n_samples": int(np.sum(fold_id_per_row == fold)),
            "n_games": int(len(np.unique(group_keys[fold_id_per_row == fold]))),
        }
        for fold in range(n_splits)
    ])

    fold_role_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "role": label_encoder.inverse_transform([r])[0],
            "count": int(np.sum(y[fold_id_per_row == fold] == r)),
        }
        for fold in range(n_splits)
        for r in np.unique(y)
    ])

    fold_dataset_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "dataset": str(ds),
            "count": int(np.sum(dataset_tags[fold_id_per_row == fold] == ds)),
        }
        for fold in range(n_splits)
        for ds in sorted(np.unique(dataset_tags))
    ])

    fold_agent_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "agent": str(ag),
            "count": int(np.sum(agent_names[fold_id_per_row == fold] == ag)),
        }
        for fold in range(n_splits)
        for ag in sorted(np.unique(agent_names))
    ])

    print("\nBalanced fold summary:")
    display(fold_summary_df)
    print("Fold x Dataset distribution:")
    display(fold_dataset_df.pivot(index="fold", columns="dataset", values="count").fillna(0).astype(int))
    print("Fold x Role distribution:")
    display(fold_role_df.pivot(index="fold", columns="role", values="count").fillna(0).astype(int))
    agent_pivot = fold_agent_df.pivot(index="fold", columns="agent", values="count").fillna(0).astype(int)
    print(f"Fold x Agent distribution (total {len(agent_pivot.columns)} agents):")
    display(agent_pivot)
    
    # 改善: POSSESSED分布の確認
    possessed_dist = fold_role_df[fold_role_df["role"] == "POSSESSED"].set_index("fold")["count"]
    print(f"\n✓ POSSESSED distribution: {dict(possessed_dist)}")
    if possessed_dist.nunique() <= 2:
        print("  ✓ POSSESSED均等配置良好 (各foldに1体配置)")

    models = {}
    cv_results = []
    # --- ここまでfold分割 ---
    for view in ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]:
        print(f"\n{'='*65}")
        print(f"  {view} 視点モデル: 訓練開始")
        print(f"{'='*65}")
        def objective(trial):
            params = {
                'objective': 'multi:softprob',
                'num_class': len(classes),
                'eval_metric': 'mlogloss',
                'tree_method': 'hist',
                'random_state': 42,
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'n_estimators': trial.suggest_int('n_estimators', 150, 800),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            }
            fold_scores = []
            for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]
                meta_val = {k: v[val_idx] for k, v in meta.items()}
                w_tr = np.array([weight_dict[label] for label in y_tr])
                model = xgb.XGBClassifier(**params)
                model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
                preds_proba = model.predict_proba(X_val)
                score, _ = evaluate_view_with_constraints(view, preds_proba, y_val, meta_val)
                fold_scores.append(score)
                trial.report(np.mean(fold_scores), step=fold)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            return float(np.mean(fold_scores))
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params.copy()
        best_params.update({
            'objective': 'multi:softprob',
            'num_class': len(classes),
            'eval_metric': 'mlogloss',
            'tree_method': 'hist',
            'random_state': 42,
        })
        fold_scores = []
        fold_samples = []
        for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}
            w_tr = np.array([weight_dict[label] for label in y_tr])
            model = xgb.XGBClassifier(**best_params)
            model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
            preds_proba = model.predict_proba(X_val)
            fold_score, n_assigned = evaluate_view_with_constraints(view, preds_proba, y_val, meta_val)
            fold_scores.append(fold_score)
            fold_samples.append(n_assigned)
            print(f"    Fold {fold}/5: assigned={n_assigned}, F1={fold_score:.4f}")
        w_full = np.array([weight_dict[label] for label in y])
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X, y, sample_weight=w_full)
        models[view] = final_model
        mean_score = float(np.mean(fold_scores)) if fold_scores else 0.0
        print(f"    ✓ {view} モデル訓練完了 (mean F1={mean_score:.4f})")
        cv_results.append({
            'model': view,
            'mean_f1': mean_score,
            'fold_scores': fold_scores,
            'assigned_samples': fold_samples,
            'best_trial': int(study.best_trial.number),
            'best_params': study.best_params,
        })
    run_status['ok'] = True
    print(f"\n{'='*70}")
    print("✓ 全モデル訓練完了")
    print(f"{'='*70}")
except Exception as e:
    run_status['error'] = str(e)
    print(f"\nTraining failed: {e}")
    print(traceback.format_exc())
finally:
    run_status['elapsed_sec'] = round(time.time() - t0, 2)
print("\nRun status:")
print(json.dumps(run_status, ensure_ascii=False, indent=2))


Balanced fold summary:


,fold,n_samples,n_games
0,1,140,28
1,2,140,28
2,3,145,29
3,4,145,29
4,5,140,28


Fold x Dataset distribution:


dataset,all_feature_table_2025sp17_with_talks,all_feature_table_2025summer_with_talks2
fold,,
1,60,80
2,60,80
3,75,70
4,70,75
5,70,70


Fold x Role distribution:


...

    Fold 1/5: assigned=136, F1=0.7353
    Fold 2/5: assigned=136, F1=0.8824
    Fold 3/5: assigned=144, F1=0.7500
    Fold 4/5: assigned=148, F1=0.7297
    Fold 5/5: assigned=124, F1=0.8065


[I 2026-04-29 12:24:52,944] A new study created in memory with name: no-name-e8f0cb47-f514-4773-9d76-170aebe673c0


    ✓ VILLAGER モデル訓練完了 (mean F1=0.7808)

  POSSESSED 視点モデル: 訓練開始


[I 2026-04-29 12:24:59,813] Trial 0 finished with value: 0.6947802197802198 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'n_estimators': 626, 'min_child_weight': 12, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 3.200866785899844e-08, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.6947802197802198.
[I 2026-04-29 12:25:08,441] Trial 1 finished with value: 0.696565934065934 and parameters: {'max_depth': 8, 'learning_rate': 0.010636066512540286, 'n_estimators': 781, 'min_child_weight': 17, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'gamma': 3.939402261362697e-07, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 1 with value: 0.696565934065934.
[I 2026-04-29 12:25:15,937] Trial 2 finished with value: 0.7644230769230769 and parameters: {'max_depth': 6, 'learning_rate': 0.023927528765580644, 'n_estimators': 548, 

    Fold 1/5: assigned=52, F1=0.8462
    Fold 2/5: assigned=56, F1=0.8571
    Fold 3/5: assigned=52, F1=0.6154
    Fold 4/5: assigned=64, F1=0.9375
    Fold 5/5: assigned=56, F1=0.8571
    ✓ POSSESSED モデル訓練完了 (mean F1=0.8227)

✓ 全モデル訓練完了

Run status:
{
  "ok": true,
  "error": null,
  "elapsed_sec": 421.48
}


In [10]:
# 追加診断: 視点ごとのCV平均と target support/F1 集計
print("="*80)
print("Quick diagnostic summary")
print("="*80)

if "cv_results" in globals() and cv_results:
    cv_df = pd.DataFrame([{
        "view": r.get("model"),
        "cv_mean_f1": r.get("mean_f1"),
        "fold_scores": r.get("fold_scores"),
    } for r in cv_results])
    display(cv_df[["view", "cv_mean_f1"]].sort_values("cv_mean_f1"))
else:
    print("cv_results がありません")

if "diag_df" in globals() and isinstance(diag_df, pd.DataFrame) and not diag_df.empty:
    display(
        diag_df.groupby(["view", "target_label"], as_index=False).agg(
            folds=("fold", "count"),
            mean_assigned_n=("assigned_n", "mean"),
            min_target_support=("target_support", "min"),
            max_target_support=("target_support", "max"),
            mean_target_f1=("target_f1", "mean"),
            std_target_f1=("target_f1", "std"),
        ).sort_values("mean_target_f1")
    )
else:
    print("diag_df がありません")

Quick diagnostic summary


,view,cv_mean_f1
1,WEREWOLF,0.485468
2,VILLAGER,0.780766
3,POSSESSED,0.822665
0,SEER,1.000000


diag_df がありません


In [11]:
# 追加表示: 各foldに含まれるファイル一覧
print("="*80)
print("Files in each fold")
print("="*80)

if "fold_id_per_row" not in globals():
    print("fold_id_per_row がありません。先に学習セルを実行してください。")
else:
    fold_file_df = pd.DataFrame({
        "fold": fold_id_per_row + 1,
        "dataset_tag": meta["dataset_tag"],
        "source_file": meta["source_file"],
    }).drop_duplicates().sort_values(["fold", "dataset_tag", "source_file"]).reset_index(drop=True)

    # foldごとの件数サマリー
    fold_summary = fold_file_df.groupby("fold", as_index=False).agg(
        n_files=("source_file", "nunique"),
        n_datasets=("dataset_tag", "nunique"),
    )
    print("\n[Summary]")
    display(fold_summary)

    # fold別ファイル一覧
    for f in sorted(fold_file_df["fold"].unique()):
        sub = fold_file_df[fold_file_df["fold"] == f].copy()
        print(f"\n[Fold {f}] files={len(sub)}")
        display(sub[["dataset_tag", "source_file"]].reset_index(drop=True))

Files in each fold

[Summary]


,fold,n_files,n_datasets
0,1,28,2
1,2,28,2
2,3,29,2
3,4,29,2
4,5,28,2



[Fold 1] files=28


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746438237_kanolab_CanisLupus_sunamelli_...
1,all_feature_table_2025sp17_with_talks,taged_1746461224_mille_kanolab_osawalab_Camell...
2,all_feature_table_2025sp17_with_talks,taged_1746463069_GAIT_ai168wolf_CanisLupus_Cam...
3,all_feature_table_2025sp17_with_talks,taged_1746480219_kanolab_mille_CanisLupus_osaw...
4,all_feature_table_2025sp17_with_talks,taged_1746485144_ai168wolf_mille_CanisLupus_os...
5,all_feature_table_2025sp17_with_talks,taged_1746511721_mille_CanisLupus_sunamelli_ka...
6,all_feature_table_2025sp17_with_talks,taged_1746527448_GPTaku_sunamelli_ai168wolf_os...
7,all_feature_table_2025sp17_with_talks,taged_1746543128_mille_CanisLupus_osawalab_sun...
8,all_feature_table_2025sp17_with_talks,taged_1746544555_GPTaku_CamelliaDragons_kanola...
9,all_feature_table_2025sp17_with_talks,taged_1746547899_GPTaku_ai168wolf_sunamelli_os...



[Fold 2] files=28


...

,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746431630_kanolab_mille_ai168wolf_osawa...
1,all_feature_table_2025sp17_with_talks,taged_1746439413_mille_CamelliaDragons_kanolab...
2,all_feature_table_2025sp17_with_talks,taged_1746441587_kanolab_CanisLupus_CamelliaDr...
3,all_feature_table_2025sp17_with_talks,taged_1746461792_CanisLupus_mille_GAIT_osawala...
4,all_feature_table_2025sp17_with_talks,taged_1746471375_kanolab_GAIT_sunamelli_osawal...
5,all_feature_table_2025sp17_with_talks,taged_1746474995_sunamelli_kanolab_GAIT_ai168w...
6,all_feature_table_2025sp17_with_talks,taged_1746482677_CamelliaDragons_mille_kanolab...
7,all_feature_table_2025sp17_with_talks,taged_1746531959_ai168wolf_osawalab_CamelliaDr...
8,all_feature_table_2025sp17_with_talks,taged_1746533146_kanolab_osawalab_sunamelli_GP...
9,all_feature_table_2025sp17_with_talks,taged_1746538054_mille_sunamelli_osawalab_Cani...



[Fold 4] files=29


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746433451_kanolab_ai168wolf_sunamelli_o...
1,all_feature_table_2025sp17_with_talks,taged_1746438728_CanisLupus_kanolab_sunamelli_...
2,all_feature_table_2025sp17_with_talks,taged_1746440914_CamelliaDragons_kanolab_mille...
3,all_feature_table_2025sp17_with_talks,taged_1746455020_CamelliaDragons_CanisLupus_os...
4,all_feature_table_2025sp17_with_talks,taged_1746464608_sunamelli_CamelliaDragons_osa...
5,all_feature_table_2025sp17_with_talks,taged_1746472566_kanolab_sunamelli_osawalab_mi...
6,all_feature_table_2025sp17_with_talks,taged_1746473490_sunamelli_ai168wolf_CamelliaD...
7,all_feature_table_2025sp17_with_talks,taged_1746529140_CamelliaDragons_sunamelli_osa...
8,all_feature_table_2025sp17_with_talks,taged_1746539847_mille_kanolab_CanisLupus_GPTa...
9,all_feature_table_2025sp17_with_talks,taged_1746548423_sunamelli_CamelliaDragons_Can...



[Fold 5] files=28


,dataset_tag,source_file
0,all_feature_table_2025sp17_with_talks,taged_1746429865_osawalab_ai168wolf_CamelliaDr...
1,all_feature_table_2025sp17_with_talks,taged_1746432378_mille_ai168wolf_osawalab_Came...
2,all_feature_table_2025sp17_with_talks,taged_1746465664_mille_sunamelli_CanisLupus_GA...
3,all_feature_table_2025sp17_with_talks,taged_1746528573_ai168wolf_GPTaku_CanisLupus_o...
4,all_feature_table_2025sp17_with_talks,taged_1746530082_GPTaku_CanisLupus_ai168wolf_m...
5,all_feature_table_2025sp17_with_talks,taged_1746535428_ai168wolf_sunamelli_mille_osa...
6,all_feature_table_2025sp17_with_talks,taged_1746540651_ai168wolf_mille_GPTaku_CanisL...
7,all_feature_table_2025sp17_with_talks,taged_1746542262_sunamelli_mille_CamelliaDrago...
8,all_feature_table_2025sp17_with_talks,taged_1746551185_CanisLupus_osawalab_mille_GPT...
9,all_feature_table_2025sp17_with_talks,taged_1746557566_mille_sunamelli_kanolab_ai168...


## 7. Fold別・モデル別の詳細評価

各フォールドで役職割り当てロジックを使用した予測結果を詳細に分析します。

In [12]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

print("="*80)
print("2DAY START: Fold別・視点別の詳細評価")
print("="*80)

if not models:
    print("❌ モデルが訓練されていません")
else:
    # 学習時のsplit_indicesを優先利用
    if 'split_indices' in globals() and split_indices is not None:
        eval_splits = split_indices
        print("Using balanced split_indices from training cell.")
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        sgkf = StratifiedGroupKFold(n_splits=5)
        eval_splits = list(sgkf.split(X, y, meta['source_file']))
        print("Using fallback StratifiedGroupKFold split.")

    for target_view in models.keys():
        print(f"\n{'='*80}")
        print(f"=== {target_view} 視点モデルの詳細評価 ===")
        print(f"{'='*80}")

        model = models[target_view]
        all_y_true_assigned = []
        all_y_pred_assigned = []

        for fold_idx, (train_idx, val_idx) in enumerate(eval_splits, start=1):
            X_val = X[val_idx]
            y_val = y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}

            y_pred_proba = model.predict_proba(X_val)
            num_games = len(y_val) // 5

            for i in range(num_games):
                s, e = i * 5, (i + 1) * 5

                if target_view == "SEER":
                    res = assign_roles_for_seer_by_divination(
                        y_pred_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        meta_val['div_result1'][s:e], meta_val['div_id1'][s:e],
                        meta_val['div_result2'][s:e], meta_val['div_id2'][s:e],
                        meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    for k in ['black', 'white']:
                        if res[k][0].size > 0:
                            all_y_pred_assigned.extend(res[k][0])
                            all_y_true_assigned.extend(res[k][1])
                else:
                    pred, true = assign_roles_for_non_seer(
                        y_pred_proba[s:e], y_val[s:e], role_counts, class_names, label_encoder,
                        target_view, meta_val['exec_id'][s:e], meta_val['attack_id'][s:e],
                        RUN_OPTIONS["day2_flag"]
                    )
                    if pred.size > 0:
                        all_y_pred_assigned.extend(pred)
                        all_y_true_assigned.extend(true)

        if len(all_y_true_assigned) > 0:
            print(f"\n評価サンプル数: {len(all_y_true_assigned)}")
            print("\n分類レポート:")
            print(classification_report(
                all_y_true_assigned,
                all_y_pred_assigned,
                labels=label_encoder.transform(class_names),
                target_names=class_names,
                zero_division=0,
            ))

            cm = confusion_matrix(
                all_y_true_assigned,
                all_y_pred_assigned,
                labels=label_encoder.transform(class_names),
            )
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            print("\n混同行列:")
            print(cm_df)
        else:
            print("評価対象サンプルなし")

2DAY START: Fold別・視点別の詳細評価
Using balanced split_indices from training cell.

=== SEER 視点モデルの詳細評価 ===

評価サンプル数: 192

分類レポート:
              precision    recall  f1-score   support

   POSSESSED       0.69      0.69      0.69        48
        SEER       0.00      0.00      0.00         0
    VILLAGER       0.84      0.84      0.84        96
    WEREWOLF       1.00      1.00      1.00        48

    accuracy                           0.84       192
   macro avg       0.63      0.63      0.63       192
weighted avg       0.84      0.84      0.84       192


混同行列:
           POSSESSED  SEER  VILLAGER  WEREWOLF
POSSESSED         33     0        15         0
SEER               0     0         0         0
VILLAGER          15     0        81         0
WEREWOLF           0     0         0        48

=== WEREWOLF 視点モデルの詳細評価 ===

評価サンプル数: 568

分類レポート:
              precision    recall  f1-score   support

   POSSESSED       0.69      0.69      0.69       142
        SEER       0.80      0.80     

## 9. 結果の保存・エクスポート

モデル、特徴量重要度、評価結果をCSV・画像・joblib形式で保存します。

In [13]:
# 実験タグ管理: old/new 分割結果を混ぜないためのラベル
EXPERIMENT_TAG = globals().get("EXPERIMENT_TAG", "new_split")
SPLIT_VERSION = globals().get("SPLIT_VERSION", "dataset_plus_source")  # 例: source_only / dataset_plus_source
print("="*80)
print("Experiment tracking")
print("="*80)
print(f"EXPERIMENT_TAG : {EXPERIMENT_TAG}")
print(f"SPLIT_VERSION  : {SPLIT_VERSION}")
print("このタグ名で保存ファイルが分離されます。")

Experiment tracking
EXPERIMENT_TAG : new_split
SPLIT_VERSION  : dataset_plus_source
このタグ名で保存ファイルが分離されます。


In [14]:
print("="*80)
print("結果の保存")
print("="*80)

output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "2day_start"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. モデルの保存
if models:
    model_path = output_dir / "models_2day_start_new_features2.joblib"
    joblib.dump(models, model_path)
    print(f"✓ Models saved: {model_path}")

# 2. 特徴量重要度の保存
if 'fi_df' in globals() and not fi_df.empty:
    fi_path = output_dir / "feature_importance_2day_start_new_features2.csv"
    fi_df.to_csv(fi_path, index=False)
    print(f"✓ Feature importance saved: {fi_path}")

# 3. balanced fold分割情報の保存
if 'fold_id_per_row' in globals():
    fold_assign_df = pd.DataFrame({
        "source_file": meta["source_file"],
        "dataset_tag": meta["dataset_tag"] if "dataset_tag" in meta else "unknown",
        "agent_name": agent_names,
        "role": label_encoder.inverse_transform(y),
        "fold": fold_id_per_row + 1,
    })
    fold_assign_path = output_dir / "fold_assignments_balanced_new_features2.csv"
    fold_assign_df.to_csv(fold_assign_path, index=False)
    print(f"✓ Fold assignments saved: {fold_assign_path}")

if 'fold_summary_df' in globals() and isinstance(fold_summary_df, pd.DataFrame):
    fold_summary_path = output_dir / "fold_split_basic_info_new_features2.csv"
    fold_summary_df.to_csv(fold_summary_path, index=False)
    print(f"✓ Fold split summary saved: {fold_summary_path}")

if 'fold_role_df' in globals() and isinstance(fold_role_df, pd.DataFrame):
    fold_role_path = output_dir / "fold_role_distribution_new_features2.csv"
    fold_role_df.to_csv(fold_role_path, index=False)
    print(f"✓ Fold role distribution saved: {fold_role_path}")

if 'fold_dataset_df' in globals() and isinstance(fold_dataset_df, pd.DataFrame):
    fold_dataset_path = output_dir / "fold_dataset_distribution_new_features2.csv"
    fold_dataset_df.to_csv(fold_dataset_path, index=False)
    print(f"✓ Fold dataset distribution saved: {fold_dataset_path}")

if 'fold_agent_df' in globals() and isinstance(fold_agent_df, pd.DataFrame):
    fold_agent_path = output_dir / "fold_agent_distribution_new_features2.csv"
    fold_agent_df.to_csv(fold_agent_path, index=False)
    print(f"✓ Fold agent distribution saved: {fold_agent_path}")

# 4. 設定とメタ情報の保存
metadata = {
    "generated_at": datetime.now().isoformat(),
    "notebook": "run_train_pipe_2day.ipynb",
    "data_paths": [str(p) for p in data_paths],
    "run_options": RUN_OPTIONS,
    "config": config,
    "models_trained": list(models.keys()) if models else [],
    "n_samples": len(X),
    "n_features": len(final_features),
    "roles": list(label_encoder.classes_),
    "cv_folds": 5,
    "split_strategy": "greedy_gamewise_balanced_by_dataset_and_agent",
}

metadata_path = output_dir / "metadata_2day_start_new_features2.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ Metadata saved: {metadata_path}")

# 5. 結果サマリー
print("\n" + "="*80)
print("2DAY START パイプライン完了")
print("="*80)
print(f"\n保存先: {output_dir}")
print(f"モデル数: {len(models) if models else 0}")
print(f"訓練サンプル数: {len(X)}")
print(f"特徴量数: {len(final_features)}")
print(f"実行時間: {run_status['elapsed_sec']} 秒")
print(f"ステータス: {'成功 ✓' if run_status['ok'] else '失敗 ✗'}")

saved_files = sorted(list(output_dir.glob("*")))
print(f"\n保存ファイル一覧:")
for f in saved_files:
    size_kb = f.stat().st_size / 1024 if f.is_file() else 0
    print(f"  - {f.name:<50} ({size_kb:>8.2f} KB)")

結果の保存
✓ Models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\models_2day_start_new_features2.joblib
✓ Fold assignments saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_assignments_balanced_new_features2.csv
✓ Fold split summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_split_basic_info_new_features2.csv
✓ Fold role distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_role_distribution_new_features2.csv
✓ Fold dataset distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_dataset_distribution_new_features2.csv
✓ Fold agent distribution saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\2day_start\fold_agent_distribution_new_features2.csv
✓ Metadata saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\note